In [2]:
import numpy as np
import pandas
import sklearn 
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from mlxtend.plotting import plot_decision_regions
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn import datasets
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [3]:
#import the data
data = pandas.read_csv('./Project.csv', delimiter=',')

In [4]:
#print the data
data

,user_id,day_type,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,burnout_score,burnout_risk
0,1,Weekday,9.59,11.86,4,2,0,7.55,91.2,19.17,Low
1,1,Weekend,7.38,10.33,4,1,0,6.69,82.0,29.70,Low
2,1,Weekend,6.31,8.92,1,2,0,8.87,80.6,32.93,Low
3,1,Weekday,8.34,10.70,4,1,1,8.13,70.0,45.47,Low
4,1,Weekend,6.97,9.83,1,2,0,5.85,67.1,51.61,Low
...,...,...,...,...,...,...,...,...,...,...,...
1795,180,Weekend,6.33,8.16,0,4,0,5.59,73.5,31.91,Low
1796,180,Weekend,4.70,7.88,0,4,0,6.69,89.8,26.30,Low
1797,180,Weekend,3.92,6.39,2,1,0,6.77,74.6,34.07,Low
1798,180,Weekday,8.93,11.11,2,5,0,8.28,74.6,38.14,Low


In [5]:
#set variables
hours_worked = data["work_hours"]
screen_time = data["screen_time_hours"]
meetings_count = data["meetings_count"]
breaks_taken = data["breaks_taken"]
after_hours_work = data["after_hours_work"]
sleep_hours = data["sleep_hours"]
task_completion = data["task_completion_rate"]
burnout_score = data["burnout_score"]
burnout_risk = data["burnout_risk"]

In [6]:
x = data.drop(columns = ['user_id','burnout_score'])
y = data['burnout_score']

In [7]:
x_nominal = pandas.get_dummies(x)

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=500)
xn_train, xn_test, yn_train, yn_test = train_test_split(x_nominal, y, test_size=500)

labels = ['model'] + x_nominal.columns.to_list()
labels
spearman_table = pandas.DataFrame(columns = labels)
spearman_table

,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium


## Linear

### Linear Regression

In [9]:
linear = LinearRegression()

In [10]:
linear.fit(xn_train, yn_train)
y_predicted_lr = linear.predict(xn_test)
mean_squared_error(yn_test, y_predicted_lr)

print("Mean Squared Error:", mean_squared_error(yn_test, y_predicted_lr))

# feature importance ranking
linear_importance = pandas.DataFrame({'feature': xn_train.columns,'importance': np.abs(linear.coef_)}).sort_values(by='importance', ascending=False)

linear_importance

Mean Squared Error: 30.143235904187257


,feature,importance
9,burnout_risk_High,21.931935
10,burnout_risk_Low,15.195464
11,burnout_risk_Medium,6.736471
6,task_completion_rate,1.334753
4,after_hours_work,0.741241
3,breaks_taken,0.184726
1,screen_time_hours,0.136304
5,sleep_hours,0.077424
0,work_hours,0.074554
2,meetings_count,0.069185


In [11]:
y_predicted_lr = pandas.DataFrame(y_predicted_lr)

In [12]:
vals = {}
vals['model'] = 'linear regression'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_predicted_lr, method='spearman')

spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

C:\Users\adria\AppData\Local\Temp\ipykernel_24484\426421780.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])


### Ridge Regression

In [13]:
#create ridge pipeline
ridge_model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0)) #alpha 1 is default, we can tune it if necessary
])

In [14]:
#train the model
ridge_model.fit(xn_train, yn_train)

Pipeline(steps=[('scaler', StandardScaler()), ('ridge', Ridge())])

In [15]:
#evaluate
y_pred = ridge_model.predict(xn_test)

print("R2 Score:", r2_score(yn_test, y_pred))
print("MSE:", mean_squared_error(yn_test, y_pred))

R2 Score: 0.9417120156437435
MSE: 30.16022797865065


In [16]:
#feature importance
#get coefficients
coefficients = ridge_model.named_steps['ridge'].coef_

#take absolute value (importance magnitude)
importance = np.abs(coefficients)

#create dataframe
ridge_importance = pandas.DataFrame({
    'feature': xn_train.columns,
    'importance': importance
})

#sort by importance
ridge_importance = ridge_importance.sort_values(by='importance', ascending=False)

ridge_importance

,feature,importance
6,task_completion_rate,20.137552
9,burnout_risk_High,3.671134
10,burnout_risk_Low,2.108092
11,burnout_risk_Medium,0.971245
4,after_hours_work,0.353622
1,screen_time_hours,0.324361
3,breaks_taken,0.260014
0,work_hours,0.164302
2,meetings_count,0.115852
5,sleep_hours,0.081600


In [17]:
#convert to ranking
ridge_importance['rank'] = ridge_importance['importance'].rank(ascending=False)

ridge_importance

,feature,importance,rank
6,task_completion_rate,20.137552,1.0
9,burnout_risk_High,3.671134,2.0
10,burnout_risk_Low,2.108092,3.0
11,burnout_risk_Medium,0.971245,4.0
4,after_hours_work,0.353622,5.0
1,screen_time_hours,0.324361,6.0
3,breaks_taken,0.260014,7.0
0,work_hours,0.164302,8.0
2,meetings_count,0.115852,9.0
5,sleep_hours,0.081600,10.0


## Decision Tree

### Regression Tree

In [18]:
tree = DecisionTreeRegressor()
parameters_tree = {# 'criterion':('squared_error','friedman_mse','absolute_error','poisson'),
                   'max_depth':[5,10,15,20],
                   'min_samples_split':[5,10,15,20],
                   'max_leaf_nodes':[5,10,20,50,None]
                  }
clf = GridSearchCV(tree, parameters_tree)
clf.fit(xn_train, yn_train)

GridSearchCV(estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': [5, 10, 15, 20],
                         'max_leaf_nodes': [5, 10, 20, 50, None],
                         'min_samples_split': [5, 10, 15, 20]})

In [19]:
y_predicted_rt = pandas.DataFrame(clf.predict(xn_test))

vals = {}
vals['model'] = 'regression tree'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_predicted_rt, method='spearman')
    
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

# feature importance ranking
tree_importance = pandas.DataFrame({'feature': xn_train.columns,'importance': clf.best_estimator_.feature_importances_}).sort_values(by='importance', ascending=False)

tree_importance

,feature,importance
6,task_completion_rate,0.950684
10,burnout_risk_Low,0.030897
9,burnout_risk_High,0.017561
5,sleep_hours,0.000858
0,work_hours,0.000000
1,screen_time_hours,0.000000
2,meetings_count,0.000000
3,breaks_taken,0.000000
4,after_hours_work,0.000000
7,day_type_Weekday,0.000000


In [20]:
spearman_table

,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium
0,linear regression,0.002768,-0.006809,-0.050083,-0.016495,0.016836,-0.067910,-0.030526,-0.039258,0.039258,-0.010474,-0.026389,0.031530
0,regression tree,0.009191,-0.000288,-0.053907,-0.018564,0.016024,-0.071879,-0.026637,-0.035407,0.035407,-0.018513,-0.021052,0.028935


In [21]:
x_nominal.columns

Index(['work_hours', 'screen_time_hours', 'meetings_count', 'breaks_taken',
       'after_hours_work', 'sleep_hours', 'task_completion_rate',
       'day_type_Weekday', 'day_type_Weekend', 'burnout_risk_High',
       'burnout_risk_Low', 'burnout_risk_Medium'],
      dtype='object')

### Random Forest

In [22]:
rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
rf_reg.fit(xn_train, yn_train)
y_pred_rf = pandas.DataFrame(rf_reg.predict(xn_test))

vals = {}
vals['model'] = 'random forest'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_pred_rf, method='spearman')
    
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

# feature importance ranking
rf_importance = pandas.DataFrame({'feature': xn_train.columns,'importance': rf_reg.feature_importances_}).sort_values(by='importance', ascending=False)

rf_importance

,feature,importance
6,task_completion_rate,0.891091
10,burnout_risk_Low,0.044414
11,burnout_risk_Medium,0.015619
5,sleep_hours,0.011341
9,burnout_risk_High,0.010264
1,screen_time_hours,0.008906
0,work_hours,0.008627
3,breaks_taken,0.003990
2,meetings_count,0.003910
4,after_hours_work,0.001306


## Boosting

### Gradient Boosting

In [23]:
gb_reg = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=3,
    random_state=42
)
gb_reg.fit(xn_train, yn_train)
y_pred_gb = pandas.DataFrame(gb_reg.predict(xn_test))

vals = {}
vals['model'] = 'gradient boosting'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_pred_gb, method='spearman')

spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

# feature importance ranking
gb_importance = pandas.DataFrame({'feature': xn_train.columns,'importance': gb_reg.feature_importances_}).sort_values(by='importance', ascending=False)

gb_importance

,feature,importance
6,task_completion_rate,0.929447
10,burnout_risk_Low,0.036194
11,burnout_risk_Medium,0.017080
9,burnout_risk_High,0.007361
1,screen_time_hours,0.003071
5,sleep_hours,0.003010
0,work_hours,0.002851
2,meetings_count,0.000426
3,breaks_taken,0.000379
4,after_hours_work,0.000179


In [24]:
spearman_table


,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium
0,linear regression,0.002768,-0.006809,-0.050083,-0.016495,0.016836,-0.067910,-0.030526,-0.039258,0.039258,-0.010474,-0.026389,0.031530
0,regression tree,0.009191,-0.000288,-0.053907,-0.018564,0.016024,-0.071879,-0.026637,-0.035407,0.035407,-0.018513,-0.021052,0.028935
0,random forest,0.005721,-0.003439,-0.055684,-0.013788,0.022073,-0.068741,-0.035042,-0.036958,0.036958,-0.006097,-0.023461,0.026834
0,gradient boosting,-0.001852,-0.011131,-0.056514,-0.016606,0.019810,-0.067665,-0.031180,-0.045051,0.045051,-0.012767,-0.024869,0.030792


### XGBoost

In [25]:
#create the model
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
# these search parameters are default/safe and can be altered

In [26]:
#train the model
xgb_model.fit(xn_train, yn_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [27]:
#evaluate performance
y_pred_xgb = xgb_model.predict(xn_test)

print("R2 Score:", r2_score(yn_test, y_pred_xgb))
print("MSE:", mean_squared_error(yn_test, y_pred_xgb))

R2 Score: 0.942787113969557
MSE: 29.60393475691537


In [28]:
#XGBoost handles feature importance in different ways. For research we use "gain" which refers to the average gain of splits.
importance_dict = xgb_model.get_booster().get_score(importance_type='gain')

In [29]:
#create dataframe with all features
xgb_importance = pandas.DataFrame({
    'feature': xn_train.columns
})

#map gain scores
xgb_importance['importance'] = xgb_importance['feature'].map(importance_dict)

#replace missing values (unused features) with 0
xgb_importance['importance'] = xgb_importance['importance'].fillna(0)

#sort
xgb_importance = xgb_importance.sort_values(by='importance', ascending=False)

xgb_importance

,feature,importance
10,burnout_risk_Low,10342.994141
6,task_completion_rate,8839.835938
9,burnout_risk_High,1297.133057
7,day_type_Weekday,96.066444
1,screen_time_hours,68.976547
4,after_hours_work,68.837036
0,work_hours,66.010658
3,breaks_taken,61.899048
5,sleep_hours,60.975315
2,meetings_count,56.726452


In [30]:
#convert to spearman ranking
xgb_importance['rank'] = xgb_importance['importance'].rank(ascending=False)

xgb_importance

,feature,importance,rank
10,burnout_risk_Low,10342.994141,1.0
6,task_completion_rate,8839.835938,2.0
9,burnout_risk_High,1297.133057,3.0
7,day_type_Weekday,96.066444,4.0
1,screen_time_hours,68.976547,5.0
4,after_hours_work,68.837036,6.0
0,work_hours,66.010658,7.0
3,breaks_taken,61.899048,8.0
5,sleep_hours,60.975315,9.0
2,meetings_count,56.726452,10.0


In [31]:
# Compare feature-importance rankings across models (Spearman correlation)
importance_tables = {
    'Linear Regression': linear_importance,
    'Ridge Regression': ridge_importance,
    'Regression Tree': tree_importance,
    'Random Forest': rf_importance,
    'Gradient Boosting': gb_importance,
    'XGBoost': xgb_importance
}

rank_table = pandas.DataFrame({
    model_name: table.set_index('feature')['importance'].rank(ascending=False, method='average')
    for model_name, table in importance_tables.items()
})

importance_rank_corr = rank_table.corr(method='spearman')
importance_rank_corr

,Linear Regression,Ridge Regression,Regression Tree,Random Forest,Gradient Boosting,XGBoost
Linear Regression,1.000000,0.930070,0.594946,0.734266,0.762238,0.468531
Ridge Regression,0.930070,1.000000,0.607427,0.741259,0.811189,0.559441
Regression Tree,0.594946,0.607427,1.000000,0.748883,0.719759,0.644871
Random Forest,0.734266,0.741259,0.748883,1.000000,0.972028,0.377622
Gradient Boosting,0.762238,0.811189,0.719759,0.972028,1.000000,0.433566
XGBoost,0.468531,0.559441,0.644871,0.377622,0.433566,1.000000
